In [1]:
import torch
from torchvision import datasets, transforms
from sklearn.metrics import mean_absolute_error
import numpy as np
from torchvision.datasets import ImageFolder  
from torch.utils.data import ConcatDataset
from torch.utils.data import random_split
from lightgbm import LGBMClassifier

In [2]:
mnist_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

dida_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1 - x),  # invert only DIDA
    transforms.Normalize((0.1307,), (0.3081,))
])

dida_data = ImageFolder(root='../datasets/dida/70000', transform=dida_transform)
mnist_data = datasets.MNIST(root='../datasets/data', train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root='../datasets/data', train=False, download=True, transform=mnist_transform)
generator = torch.Generator().manual_seed(42)

dida_train, dida_test = random_split(dida_data, [0.8, 0.2], generator=generator)
train_data = ConcatDataset([mnist_data, dida_train])
test_data = ConcatDataset([mnist_test, dida_test])

X_train = np.array([img.numpy().flatten() for img, label in train_data])  # flatten 28x28 → 784
y_train = np.array([label for img, label in train_data])
X_test = np.array([img.numpy().flatten() for img, label in test_data])
y_test = np.array([label for img, label in test_data])

idx = np.random.permutation(len(X_train))
X_train, y_train = X_train[idx], y_train[idx]


In [3]:
model = LGBMClassifier(random_state=42, n_estimators=100, force_col_wise=True)
model.fit(X_train, y_train)
pred = model.predict(X_test)
mae = mean_absolute_error(pred, y_test)
print("Loss: {}".format(mae))

[LightGBM] [Info] Total Bins 191469
[LightGBM] [Info] Number of data points in the train set: 116000, number of used features: 784
[LightGBM] [Info] Start training from score -2.307511
[LightGBM] [Info] Start training from score -2.242204
[LightGBM] [Info] Start training from score -2.300776
[LightGBM] [Info] Start training from score -2.297083
[LightGBM] [Info] Start training from score -2.319714
[LightGBM] [Info] Start training from score -2.356241
[LightGBM] [Info] Start training from score -2.306558
[LightGBM] [Info] Start training from score -2.281347
[LightGBM] [Info] Start training from score -2.314553
[LightGBM] [Info] Start training from score -2.303620


c:\Users\jake_\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Loss: 0.24825


In [4]:
accuracy = model.score(X_test, y_test)
print('Accuracy: {:.2f}%'.format(accuracy * 100))

c:\Users\jake_\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 93.12%
